In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
import config

try:
    config.assert_data_exists()
    print("✓ Data path:", config.DATA_ROOT)
except FileNotFoundError as e:
    print("✗ Data path error:", e)

import pandas as pd
import numpy as np

def load_csv(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].str.strip()
    return df

✓ Data path: /home/hareee234/Dev/sjsu/cmpe188-final-proj/flight-delay-proj-data


# 02 — Feature Engineering (Part 2: BTS 2023)
**CMPE 188 | Flight Delay Prediction**

Unlike Part 1, the Part 2 dataset already contains daily weather and aircraft info.
This notebook focuses on **derived features** (computed from existing columns)
and **target encodings** (computed on train-split only to prevent leakage).

**Set `SAMPLE_SIZE = None` below to run on the full 6.7M rows.**

Derived features planned:
- Date parsing: `FlightDate` → `month`, `day_of_month`, `season`, `is_weekend`
- Target encodings (train-only): `airline_delay_rate`, `airport_delay_rate`, `route_delay_rate`, `manufacturer_delay_rate`
- Volume features: `route_volume`, `dep_airport_volume`
- Binning: `aircraft_age_bucket`
- Ratio features: `dep_delay_carrier_pct`

In [2]:
# ─────────────────────────────────────────────────────
# Configurable sample size (None = full 6.7M rows)
SAMPLE_SIZE = 500_000
# ─────────────────────────────────────────────────────

DATA_PATH = str(config.DATA_PART2_PROCESSED / 'flights_2023_merged.csv')

if SAMPLE_SIZE:
    print(f"Loading {SAMPLE_SIZE:,} rows...")
    df = pd.read_csv(DATA_PATH, nrows=SAMPLE_SIZE)
else:
    print("Loading full 6.7M dataset...")
    df = pd.read_csv(DATA_PATH)

df.columns = df.columns.str.strip()
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].str.strip()

print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} cols")
df.head(3)

Loading 500,000 rows...


/tmp/ipykernel_4407/87314222.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=["object"]).columns:


Loaded: 500,000 rows x 52 cols


,FlightDate,Day_Of_Week,Airline,Tail_Number,Dep_Airport,Dep_CityName,DepTime_label,Dep_Delay,Dep_Delay_Tag,Dep_Delay_Type,...,dep_STATE,dep_COUNTRY,dep_LATITUDE,dep_LONGITUDE,arr_AIRPORT,arr_CITY,arr_STATE,arr_COUNTRY,arr_LATITUDE,arr_LONGITUDE
0,2023-01-02,1,Endeavor Air,N605LR,BDL,"Hartford, CT",Morning,-3,0,Low <5min,...,CT,USA,41.93887,-72.68323,LaGuardia Airport (Marine Air Terminal),New York,NY,USA,40.77724,-73.87261
1,2023-01-03,2,Endeavor Air,N605LR,BDL,"Hartford, CT",Morning,-5,0,Low <5min,...,CT,USA,41.93887,-72.68323,LaGuardia Airport (Marine Air Terminal),New York,NY,USA,40.77724,-73.87261
2,2023-01-04,3,Endeavor Air,N331PQ,BDL,"Hartford, CT",Morning,-5,0,Low <5min,...,CT,USA,41.93887,-72.68323,LaGuardia Airport (Marine Air Terminal),New York,NY,USA,40.77724,-73.87261


## 1. Date Parsing & Time Features

In [3]:
# Parse FlightDate to extract temporal features
df["FlightDate"] = pd.to_datetime(df["FlightDate"])
df["month"] = df["FlightDate"].dt.month
df["day_of_month"] = df["FlightDate"].dt.day
df["quarter"] = df["FlightDate"].dt.quarter
df["is_weekend"] = df["Day_Of_Week"].isin([6, 7]).astype(int)

# Season: 1=Winter(Dec-Feb), 2=Spring(Mar-May), 3=Summer(Jun-Aug), 4=Fall(Sep-Nov)
season_map = {12: 1, 1: 1, 2: 1, 3: 2, 4: 2, 5: 2, 6: 3, 7: 3, 8: 3, 9: 4, 10: 4, 11: 4}
df["season"] = df["month"].map(season_map)
season_labels = {1: "Winter", 2: "Spring", 3: "Summer", 4: "Fall"}
df["season_label"] = df["season"].map(season_labels)

print(f"Date range: {df['FlightDate'].min().date()} to {df['FlightDate'].max().date()}")
print(f"Months present: {sorted(df['month'].unique())}")
print(f"Quarters:       {sorted(df['quarter'].unique())}")
print()
print("Season distribution:")
print(df["season_label"].value_counts().to_string())

Date range: 2023-01-01 to 2023-01-31
Months present: [np.int32(1)]
Quarters:       [np.int32(1)]

Season distribution:
season_label
Winter    500000


In [4]:
# Encode DepTime_label as ordinal numeric
time_order = {"Morning": 0, "Afternoon": 1, "Evening": 2, "Night": 3}
df["dep_time_ordinal"] = df["DepTime_label"].map(time_order)

# Aircraft age bucket
df["aircraft_age_bucket"] = pd.cut(
    df["Aicraft_age"],
    bins=[0, 5, 15, 25, 60],
    labels=["0-5y", "5-15y", "15-25y", "25y+"],
    right=True,
).astype(str)
print("Aircraft age bucket distribution:")
print(df["aircraft_age_bucket"].value_counts().to_string())

Aircraft age bucket distribution:
aircraft_age_bucket
5-15y     203025
15-25y    197697
0-5y       79256
25y+       20022


## 2. Train/Test Split (Before Derived Features)

Split early to prevent target leakage from derived rate features.

In [5]:
from sklearn.model_selection import train_test_split

target = "Dep_Delay_Tag"

# Drop columns that are either identifiers, leakage-prone, or redundant
drop_cols = [
    "Tail_Number",           # High cardinality, identifier
    "FlightDate",            # Parsed into month/season/day
    "Dep_Delay",             # Leakage: delay amount (target is Dep_Delay_Tag)
    "Arr_Delay",             # Leakage: arrival delay knows departure delay
    "Arr_Delay_Type",        # Leakage
    "Dep_Delay_Type",        # Leakage: derived from Dep_Delay
    "Dep_CityName",          # Redundant with Dep_Airport
    "Arr_CityName",          # Redundant with Arr_Airport
    "dep_AIRPORT",           # Full name, use Dep_Airport instead
    "arr_AIRPORT",           # Full name, use Arr_Airport instead
    "dep_CITY",              # Redundant with dep_STATE
    "arr_CITY",              # Redundant with arr_STATE
    "dep_COUNTRY",           # Always USA
    "arr_COUNTRY",           # Always USA
]

drop_cols = [c for c in drop_cols if c in df.columns]
X = df.drop(columns=drop_cols + [target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"After dropping: {X.shape[1]} features")
print(f"Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}")
print(f"Train delay rate: {y_train.mean():.3f}  |  Test delay rate: {y_test.mean():.3f}")
print(f"\nDropped columns ({len(drop_cols)}):")
for c in drop_cols:
    print(f"  - {c}")

After dropping: 45 features
Train: 400,000  |  Test: 100,000
Train delay rate: 0.382  |  Test delay rate: 0.382

Dropped columns (14):
  - Tail_Number
  - FlightDate
  - Dep_Delay
  - Arr_Delay
  - Arr_Delay_Type
  - Dep_Delay_Type
  - Dep_CityName
  - Arr_CityName
  - dep_AIRPORT
  - arr_AIRPORT
  - dep_CITY
  - arr_CITY
  - dep_COUNTRY
  - arr_COUNTRY


## 3. Target-Encoded Rate Features (Train-Only, No Leakage)

In [6]:
global_rate = y_train.mean()
X_train = X_train.copy()
X_test = X_test.copy()

# --- Airline delay rate ---
airline_rate = X_train.join(y_train).groupby("Airline")[target].mean()
X_train["airline_delay_rate"] = X_train["Airline"].map(airline_rate).fillna(global_rate)
X_test["airline_delay_rate"] = X_test["Airline"].map(airline_rate).fillna(global_rate)

# --- Origin airport delay rate ---
dep_rate = X_train.join(y_train).groupby("Dep_Airport")[target].mean()
X_train["dep_airport_delay_rate"] = X_train["Dep_Airport"].map(dep_rate).fillna(global_rate)
X_test["dep_airport_delay_rate"] = X_test["Dep_Airport"].map(dep_rate).fillna(global_rate)

# --- Route delay rate (origin-destination pair) ---
route_rate = X_train.join(y_train).groupby(["Dep_Airport", "Arr_Airport"])[target].mean()
route_keys_train = list(zip(X_train["Dep_Airport"], X_train["Arr_Airport"]))
route_keys_test = list(zip(X_test["Dep_Airport"], X_test["Arr_Airport"]))
X_train["route_delay_rate"] = [route_rate.get(k, global_rate) for k in route_keys_train]
X_test["route_delay_rate"] = [route_rate.get(k, global_rate) for k in route_keys_test]

# --- Manufacturer delay rate ---
mfr_rate = X_train.join(y_train).groupby("Manufacturer")[target].mean()
X_train["manufacturer_delay_rate"] = X_train["Manufacturer"].map(mfr_rate).fillna(global_rate)
X_test["manufacturer_delay_rate"] = X_test["Manufacturer"].map(mfr_rate).fillna(global_rate)

print("Target-encoded features added.")
print(f"Global delay rate (train): {global_rate:.4f}")
print(f"\nAirline delay rate range:   {X_train['airline_delay_rate'].min():.3f} - {X_train['airline_delay_rate'].max():.3f}")
print(f"Airport delay rate range:   {X_train['dep_airport_delay_rate'].min():.3f} - {X_train['dep_airport_delay_rate'].max():.3f}")
print(f"Route delay rate range:     {X_train['route_delay_rate'].min():.3f} - {X_train['route_delay_rate'].max():.3f}")
print(f"Mfr delay rate range:       {X_train['manufacturer_delay_rate'].min():.3f} - {X_train['manufacturer_delay_rate'].max():.3f}")

Target-encoded features added.
Global delay rate (train): 0.3822

Airline delay rate range:   0.203 - 0.522
Airport delay rate range:   0.000 - 0.786
Route delay rate range:     0.000 - 1.000
Mfr delay rate range:       0.279 - 0.420


## 4. Volume & Ratio Features

In [7]:
# Route volume (number of flights per origin-destination pair)
route_counts = X_train.groupby(["Dep_Airport", "Arr_Airport"]).size().reset_index(name="route_volume")
route_counts_dict = dict(zip(zip(route_counts["Dep_Airport"], route_counts["Arr_Airport"]), route_counts["route_volume"]))

X_train["route_volume"] = [route_counts_dict.get((d, a), 1) for d, a in zip(X_train["Dep_Airport"], X_train["Arr_Airport"])]
X_test["route_volume"] = [route_counts_dict.get((d, a), 1) for d, a in zip(X_test["Dep_Airport"], X_test["Arr_Airport"])]

# Origin airport traffic volume
dep_counts = X_train.groupby("Dep_Airport").size()
X_train["dep_airport_volume"] = X_train["Dep_Airport"].map(dep_counts).fillna(1)
X_test["dep_airport_volume"] = X_test["Dep_Airport"].map(dep_counts).fillna(1)

# Delay cause ratios (only meaningful when Dep_Delay > 0, but keep as features)
for cause in ["Delay_Carrier", "Delay_Weather", "Delay_NAS", "Delay_LastAircraft"]:
    if cause in X_train.columns:
        # Avoid division by zero; Dep_Delay was dropped so we use absolute carrier delay / flight duration as proxy
        col_name = cause.replace("Delay_", "ratio_").lower()
        if "Flight_Duration" in X_train.columns:
            X_train[col_name] = X_train[cause] / X_train["Flight_Duration"].replace(0, np.nan)
            X_test[col_name] = X_test[cause] / X_test["Flight_Duration"].replace(0, np.nan)
            X_train[col_name] = X_train[col_name].fillna(0)
            X_test[col_name] = X_test[col_name].fillna(0)

print(f"Added volume features: route_volume, dep_airport_volume")
print(f"Route volume range: {X_train['route_volume'].min()} - {X_train['route_volume'].max()}")
print(f"Dep airport vol range: {X_train['dep_airport_volume'].min()} - {X_train['dep_airport_volume'].max()}")
print(f"\nFinal feature count: {X_train.shape[1]}")

Added volume features: route_volume, dep_airport_volume
Route volume range: 1 - 798
Dep airport vol range: 2 - 20882

Final feature count: 55


## 5. Column Classification & Preprocessing Pipeline

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2

# Categorical: low-to-medium cardinality categorical cols for OHE
categorical_cols = [
    "Airline", "Dep_Airport", "Arr_Airport",
    "DepTime_label", "Distance_type",
    "Manufacturer", "Model",
    "dep_STATE", "arr_STATE",
    "season_label", "aircraft_age_bucket",
]

# Drop Day_Of_Week as categorical (it's >= 5 giving issues with chi2 + SelectKBest)
# Instead keep it as numeric

# Numeric: all remaining numeric columns (including engineered ones)
exclude_from_numeric = categorical_cols + [target] + drop_cols

numeric_cols = [c for c in X_train.columns
                if X_train[c].dtype in ["int64", "float64"]
                and c not in categorical_cols]

# Keep only columns that exist
categorical_cols = [c for c in categorical_cols if c in X_train.columns]
numeric_cols = [c for c in numeric_cols if c in X_train.columns]

# Drop columns that are in neither list (should be 0)
used = set(categorical_cols + numeric_cols)
unused = [c for c in X_train.columns if c not in used]

# Build preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
        ("num", MinMaxScaler(), numeric_cols),
    ]
)

# SelectKBest with higher k for more features
selector = SelectKBest(score_func=chi2, k=80)

print(f"Categorical ({len(categorical_cols)}):")
for c in categorical_cols:
    print(f"  {c} (unique: {X_train[c].nunique()})")

print(f"\nNumeric ({len(numeric_cols)}):")
for c in numeric_cols[:15]:
    print(f"  {c}")
if len(numeric_cols) > 15:
    print(f"  ... and {len(numeric_cols) - 15} more")

if unused:
    print(f"\nUnused columns ({len(unused)}): {unused}")
else:
    print("\n✓ All columns assigned to categorical or numeric")

Categorical (11):
  Airline (unique: 15)
  Dep_Airport (unique: 339)
  Arr_Airport (unique: 339)
  DepTime_label (unique: 4)
  Distance_type (unique: 3)
  Manufacturer (unique: 4)
  Model (unique: 19)
  dep_STATE (unique: 54)
  arr_STATE (unique: 54)
  season_label (unique: 1)
  aircraft_age_bucket (unique: 4)

Numeric (41):
  Day_Of_Week
  Flight_Duration
  Delay_Carrier
  Delay_Weather
  Delay_NAS
  Delay_Security
  Delay_LastAircraft
  Aicraft_age
  dep_tavg
  dep_tmin
  dep_tmax
  dep_prcp
  dep_snow
  dep_wdir
  dep_wspd
  ... and 26 more

Unused columns (3): ['month', 'day_of_month', 'quarter']


## 6. Verify Preprocessing (Fit-Transform a Small Sample)

In [9]:
# Quick transformation check on a tiny sample
X_sample = X_train.head(1000)
X_transformed = preprocessor.fit_transform(X_sample)
print(f"Preprocessed shape: {X_transformed.shape}")
print(f"Any NaN after transform: {np.isnan(X_transformed).any()}")
print(f"Sparsity: {(X_transformed == 0).mean():.2%} zeros")
print("\n✓ Preprocessing pipeline works")

Preprocessed shape: (1000, 477)
Any NaN after transform: False
Sparsity: 91.97% zeros

✓ Preprocessing pipeline works


## 7. Save Processed Data

Save the feature-engineered train/test splits for the modeling notebooks.
Columns not included in categorical/numeric lists are dropped before save.

In [10]:
import pickle

# Save train/test splits (with engineered features)
train_data = {
    "X_train": X_train,
    "X_test": X_test,
    "y_train": y_train,
    "y_test": y_test,
    "categorical_cols": categorical_cols,
    "numeric_cols": numeric_cols,
}

output_path = config.DATA_PART2_PROCESSED / "part2_train_test_split.pkl"
with open(output_path, "wb") as f:
    pickle.dump(train_data, f)

print(f"✓ Saved train/test data to {output_path}")
print(f"  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")
print(f"  Target encoded features: airline_delay_rate, dep_airport_delay_rate, route_delay_rate, manufacturer_delay_rate")
print(f"\nNext: Run 03_model_baseline.ipynb")

✓ Saved train/test data to /home/hareee234/Dev/sjsu/cmpe188-final-proj/flight-delay-proj-data/part2/processed/part2_train_test_split.pkl
  X_train: (400000, 55)
  X_test:  (100000, 55)
  Target encoded features: airline_delay_rate, dep_airport_delay_rate, route_delay_rate, manufacturer_delay_rate

Next: Run 03_model_baseline.ipynb
